In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import boto3
from io import BytesIO

try:
    import seaborn as sns
except:
    ! pip install seaborn
    import seaborn as sns

try:
    from mlxtend.preprocessing import TransactionEncoder
except:
    ! pip install mlxtend
    from mlxtend.preprocessing import TransactionEncoder

  Using cached mlxtend-0.24.0-py3-none-any.whl.metadata (7.3 kB)
  Using cached numpy-2.4.3.tar.gz (20.7 MB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... error
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      + /home/ec2-user/anaconda3/envs/python3/bin/python3.12 /tmp/pip-install-ig1ncj92/numpy_a83d5dd728194b7abaa5257936c9645a/vendored-meson/meson/meson.py setup /tmp/pip-install-ig1ncj92/numpy_a83d5dd728194b7abaa5257936c9645a /tmp/pip-install-ig1ncj92/numpy_a83d5dd728194b7abaa5257936c9645a/.mesonpy-n1v06tcu -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=/tmp/pip-install-ig1ncj92/numpy_a83d5dd728194b7abaa5257936c9645a/.mesonpy-n1v06tcu/meson-python-native-file.ini
      The Meson build system
      Version: 1.9.2
      Source dir: /t

ModuleNotFoundError: No module named 'mlxtend'

## Helper Classes

In [ ]:
class TransactionPreprocessor:
    def __init__(self, str_uri, str_bucket, str_dirname_output):
        self.str_uri = str_uri
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_data = None
        self.list_transactions = None
        self.df_basket = None

    def import_data(self):
        self.df_data = pd.read_parquet(self.str_uri)
        print(f'Data loaded: {self.df_data.shape[0]:,} rows, {self.df_data.shape[1]} columns')
        return self

    def build_transactions(self):
        print('\nBuilding transaction lists...')
        # group items by (member_id, transaction_date) to form baskets
        self.list_transactions = (
            self.df_data
            .groupby(['member_id', 'transaction_date'])['item']
            .apply(list)
            .tolist()
        )
        # remove duplicate items within each transaction
        self.list_transactions = [list(set(t)) for t in self.list_transactions]
        # filter out single-item transactions (no associations possible)
        int_before = len(self.list_transactions)
        self.list_transactions = [t for t in self.list_transactions if len(t) >= 2]
        int_after = len(self.list_transactions)
        print(f'Total transactions: {int_before:,}')
        print(f'Single-item transactions removed: {int_before - int_after:,}')
        print(f'Multi-item transactions retained: {int_after:,}')
        return self

    def encode_transactions(self):
        print('\nOne-hot encoding transactions...')
        cls_encoder = TransactionEncoder()
        arr_encoded = cls_encoder.fit_transform(self.list_transactions)
        self.df_basket = pd.DataFrame(arr_encoded, columns=cls_encoder.columns_)
        print(f'Basket matrix shape: {self.df_basket.shape}')
        print(f'Transactions: {self.df_basket.shape[0]:,}')
        print(f'Unique items: {self.df_basket.shape[1]:,}')
        return self

    def plot_support_distribution(self):
        ser_support = self.df_basket.mean().sort_values(ascending=False)
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        # top 30 items by support
        ser_support.head(30).plot(kind='barh', ax=axes[0], color='#4C72B0', edgecolor='black')
        axes[0].set_title('Top 30 Items by Support', fontsize=14, y=1.02)
        axes[0].set_xlabel('Support (Proportion of Transactions)')
        axes[0].invert_yaxis()
        # support distribution
        axes[1].hist(ser_support, bins=50, color='#4C72B0', edgecolor='black', alpha=0.8)
        axes[1].set_title('Item Support Distribution', fontsize=14, y=1.02)
        axes[1].set_xlabel('Support')
        axes[1].set_ylabel('Number of Items')
        axes[1].set_yscale('log')
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_support_distribution.png', bbox_inches='tight', dpi=150)
        plt.show()
        return self

    def save_to_s3(self):
        # save basket matrix
        str_s3_key = '02_preprocessing/basket_matrix.parquet'
        print(f'\nSaving basket matrix to S3: s3://{self.str_bucket}/{str_s3_key}')
        buffer = BytesIO()
        self.df_basket.to_parquet(buffer, index=False)
        buffer.seek(0)
        try:
            self.s3_client.put_object(
                Bucket=self.str_bucket,
                Key=str_s3_key,
                Body=buffer.getvalue()
            )
            print(f'Successfully uploaded basket matrix to S3')
        except Exception as e:
            print(f'Error uploading to S3: {e}')
        return self

    def print_summary(self):
        print('\n=== PREPROCESSING SUMMARY ===')
        print(f'Multi-item transactions: {len(self.list_transactions):,}')
        print(f'Unique items in basket matrix: {self.df_basket.shape[1]:,}')
        ser_support = self.df_basket.mean()
        print(f'Mean item support: {ser_support.mean():.4f}')
        print(f'Median item support: {ser_support.median():.4f}')
        print(f'Max item support: {ser_support.max():.4f} ({ser_support.idxmax()})')
        print(f'Min item support: {ser_support.min():.4f} ({ser_support.idxmin()})')
        flt_sparsity = 1.0 - self.df_basket.values.mean()
        print(f'Basket matrix sparsity: {flt_sparsity:.4f} ({flt_sparsity*100:.2f}%)')
        return self

## Constants

In [ ]:
str_bucket = 'market-basket-analysis-demo'
str_task = '02_preprocessing'
str_dirname_output = './output'
str_uri = f's3://{str_bucket}/00_data_collection/groceries.parquet'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f'Output directory ready: {str_dirname_output}')

## Run Preprocessing

In [ ]:
cls_preprocessor = TransactionPreprocessor(str_uri, str_bucket, str_dirname_output)
cls_preprocessor.import_data()

In [ ]:
cls_preprocessor.build_transactions()

In [ ]:
cls_preprocessor.encode_transactions()

In [ ]:
cls_preprocessor.plot_support_distribution()

In [ ]:
cls_preprocessor.print_summary()

In [ ]:
cls_preprocessor.save_to_s3()

## Completion

In [ ]:
print('\n=== PREPROCESSING COMPLETE ===')
print(f'Basket matrix ready for next stage: s3://{str_bucket}/02_preprocessing/basket_matrix.parquet')